# Upload Data to TIDBRepo

## Module-Loading

In [1]:

from dbrepo.RestClient import RestClient
import urllib3
import pandas as pd


In [2]:

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [3]:

from dotenv import load_dotenv
import os

load_dotenv()


True

## Create Client
Create a username and password on https://tidbrepo.usm.my if you do not already have one. You should be able to create a container ID when you register.

In [ ]:

client = RestClient(
    endpoint = "https://tidbrepo.usm.my",
    username = os.getenv("DBREPO_USER"),
    password = os.g,
    secure = False
)
    

### List the container ID

In [ ]:
client.get_containers()

In [ ]:

cont_id = client.get_containers()[0].id


## Create the Database

You can create the database programmatically, but an error will show up even though the database has been created.

In [ ]:

#db = client.create_database(
#    name = "env_forensics_data",
#    container_id = cont_id,
#    is_public = True
#)


List all databases. Confirm that the database has been created.

In [ ]:

dbs = client.get_databases()
for d in dbs:
    print(d.name, d.id)
    

In [ ]:

db_id = client.get_databases()[0].id
print(db_id)


### Load the data

In [ ]:

df = pd.read_csv('class_honey_dat.csv')
print(df.head(10))


#### Standardize column names

In [ ]:

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("/","_", regex=False)
    .str.replace(" ","_", regex=False)
)
print(df.head(10))


#### Set index (primary key)

In [ ]:

df = df.set_index("sample_id")
df.index.name = "sample_id"


### Create Table without Data

In [ ]:

table = client.create_table(
    database_id=db_id,
    name="honey_elemental_analysis",
    is_public=True,
    is_schema_public=True,
    dataframe=df,
    description="Elemental analysis of honey samples by ICP OES.",
    with_data=False
)


### Check Table ID

In [ ]:

tables = client.get_tables(db_id)

for t in tables:
    print(t.name, t.id)


### Assign Table ID

In [ ]:

table_id = next(t.id for t in tables if t.name == "honey_elemental_analysis")
print(table_id)


#### Insert Rows

##### Clean the Data First

In [ ]:

import pandas as pd
import numpy as np

df_insert = df.reset_index()

# replace inf/-inf first
df_insert = df_insert.replace([np.inf, -np.inf], None)

# then replace NaN with None, using the same dataframe as the mask
df_insert = df_insert.where(pd.notnull(df_insert), None)

print(df_insert.head(10))


#### Insert the Data Row-by-Row

In [ ]:

failed_rows = []

for i, row in df_insert.iterrows():
    try:
        data = {
            k: (v.item() if hasattr(v, "item") else v)
            for k, v in row.to_dict().items()
        }

        client.create_table_data(
            database_id=db_id,
            table_id=table_id,
            data=data
        )

        if i % 10 == 0:
            print(f"Inserted row {i}")

    except Exception as e:
        failed_rows.append((i, str(e)))
        print(f"Failed row {i}: {e}")

print("Done")
print("Failed rows:", len(failed_rows))


## Publish the Identifier (PID)

In [ ]:
import requests

payload = {
    "database_id": db_id,
    "type": "database",
    "titles": [
        {
            "title": "Environmental Forensics Data: Honey Elemental Analysis",
            "language": "en",
            "type": None
        }
    ],
    "descriptions": [
        {
            "description": "Elemental analysis of honey samples using ICP OES. Data provided by Dr. Syahidah Akmal Muhammad.",
            "language": "en",
            "type": "Abstract"
        },
        {
            "description": "Measured by ICP OES. Samples of honey.",
            "language": "en",
            "type": "Methods"
        },
        {
            "description": "Multiple columns of elemental concentrations of honey samples with locations.",
            "language": "en",
            "type": "Other"
        }
    ],
    "funders": [],
    "publisher": "School of Industrial Technology, Universiti Sains Malaysia",
    "licenses": [
        {
            "identifier": "CC0-1.0",
            "uri": "https://creativecommons.org/publicdomain/zero/1.0/legalcode",
            "description": "CC0 waives copyright interest in a work you've created and dedicates it to the world-wide public domain. Use CC0 to opt out of copyright entirely and ensure your work has the widest reach."
        }
    ],
    "creators": [
        {
            "firstname": "Yusri",
            "lastname": "Yusup",
            "creator_name": "Yusup, Yusri",
            "name_type": "Personal",
            "name_identifier": "https://orcid.org/0000-0001-6703-2208",
            "affiliation": "Universiti Sains Malaysia",
            "affiliation_identifier": None
        }
    ],
    "publication_day": 10,
    "publication_month": 4,
    "publication_year": 2026,
    "related_identifiers": []
}

response = requests.post(
    f"{client.endpoint}/api/v1/identifier",
    auth=(client.username, client.password),
    verify=client.secure,
    headers={"Accept": "application/json", "Content-Type": "application/json"},
    json=payload
)

print(response.status_code)
print(response.text)

### Publish It

Publish it on the website GUI.

## Add Semantics to the Table

In [ ]:
table_meta = client.get_table(db_id,table_id)

In [ ]:
col_map = {col.internal_name: col.id for col in table_meta.columns}
print(col_map)

In [ ]:
client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["latitude"],
    concept_uri="http://dbpedia.org/ontology/latitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["longitude"],
    concept_uri="http://dbpedia.org/ontology/longitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree",
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["latitude"],
    concept_uri="http://dbpedia.org/ontology/latitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

## Identifiers / Categorical

In [ ]:

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["sample_id"],
    concept_uri="http://dbpedia.org/ontology/identifier",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["floral"],
    concept_uri="http://dbpedia.org/ontology/Plant",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["country"],
    concept_uri="http://dbpedia.org/ontology/Country",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["island"],
    concept_uri="http://dbpedia.org/ontology/Island",
    unit_uri=None
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["year"],
    concept_uri="http://dbpedia.org/ontology/date",
    unit_uri=None
)

# --- spatial ---
client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["latitude"],
    concept_uri="http://dbpedia.org/ontology/latitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["longitude"],
    concept_uri="http://dbpedia.org/ontology/longitude",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/degree"
)

# --- elemental concentrations (ppm) ---
ppm = "http://www.ontology-of-units-of-measure.org/resource/om-2/partsPerMillion"

client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["li"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["b"],  concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["na"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["mg"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["al"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["k"],  concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ca"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["v"],  concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["cr"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["mn"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["fe"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["co"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ni"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["cu"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["zn"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ga"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["rb"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["sr"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ag"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["cs"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ba"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["la"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["ce"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)
client.update_table_column(database_id=db_id, table_id=table_id, column_id=col_map["pb"], concept_uri="http://dbpedia.org/ontology/ChemicalSubstance", unit_uri=ppm)

# --- ratio ---
client.update_table_column(
    database_id=db_id,
    table_id=table_id,
    column_id=col_map["rb_sr"],
    concept_uri="http://dbpedia.org/ontology/ratio",
    unit_uri="http://www.ontology-of-units-of-measure.org/resource/om-2/one"
)